In [ ]:
import numpy as np
from matplotlib import pyplot as plt
import pandas as pd

import statsmodels.api as sm

from ISLP.models import (ModelSpec as MS, summarize , poly)
from ISLP import confusion_table
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from sklearn.neighbors import KNeighborsClassifier

# QDA

## Load Data

In [ ]:
default = pd.read_csv("../data/Default.csv")
default['student'] = default['student'].astype('category')
default['default'] = default['default'].astype('category')
default.head()

# set seed for reproducibility
train_df = default.sample(frac=0.7, random_state=123)   # 70% train
test_df  = default.drop(train_df.index)                 # 30% test

design = MS(['balance'], intercept=False) # intercept already handled in LDA/QDA
X_train = design.fit_transform(train_df)

X_test = design.transform(test_df)


## Fit

In [ ]:
qda = QDA(store_covariance=True) # by default, we don't store the covariance matrices
qda.fit(X_train, train_df['default'])

In [ ]:
qda.classes_

In [ ]:
qda.means_

In [ ]:
qda.priors_

In [ ]:
qda.covariance_

In [ ]:
qda.predict(X_train.head())

If we want the calss probabilities:

In [ ]:
qda.predict_proba(X_train.head())

## Performance

In [ ]:
train_pred = qda.predict(X_train)
confusion_table(train_df['default'], train_pred)

In [ ]:
np.mean(train_df['default']!= train_pred)

In [ ]:
test_pred = qda.predict(X_test)
confusion_table(test_df['default'], test_pred)

In [ ]:
np.mean(test_df['default']!= test_pred)

# KNN

Experiment with the detault data:

## Fit

In [ ]:
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, train_df['default'])

In [ ]:
pred_train_knn = knn.predict(X_train)

In [ ]:
pred_train_knn

Get class probabilities:

In [ ]:
knn.predict_proba(X_test.head())

## Performance

In [ ]:
confusion_table(train_df['default'], pred_train_knn)

In [ ]:
np.mean(pred_train_knn != train_df['default'])

In [ ]:
pred_test_knn = knn.predict(X_test)
confusion_table(test_df['default'], pred_test_knn)


In [ ]:
np.mean(pred_test_knn != test_df['default'])

# Multiclass Comparison
This example compares a Gaussian mixture in 2D with three mixture components.

## Generate Data

In [ ]:
np.random.seed(123)

n = 10**4
weights = np.array([0.25, 0.5, 0.25])

means = np.array([
    [-1.0,  -2.0],
    [ -1.0,  1.0],
    [ 2.0, 2.0]
])

covs = np.array([
    [[1.0,  0.3],
     [0.3,  1.0]],
    [[1.0,  0.0],
     [0.0,  1.0]],
    [[1.0,  0.0],
     [0.0,  4.0]]
])

# component labels
z = np.random.choice(3, size=n, p=weights)
x = np.zeros((n, 2))
for i in range(n):
    x[i,:] =np.random.multivariate_normal(means[z[i]], covs[z[i]])

df = pd.DataFrame(x, columns=['x1', 'x2'])
df['class'] = pd.Categorical(z)
df.head()

In [ ]:
fig, axes = plt.subplots()
for k, g in df.groupby("class"):
    g.plot.scatter("x1", "x2", s=10, alpha=0.5, label=f"Class {k}", ax=axes, color=f"C{k}")
axes.set_title("3-Component 2D Gaussian Mixture")
axes.set_xlabel(r"$x_1$")
axes.set_ylabel(r"$x_2$")
axes.legend()

In [ ]:

# set seed for reproducibility
train_df = df.sample(frac=0.7, random_state=123)   # 70% train
test_df  = df.drop(train_df.index)                 # 30% test

design = MS(['x1', 'x2'], intercept=False) # intercept already handled in LDA/QDA
X_train = design.fit_transform(train_df)

X_test = design.transform(test_df)


## LDA and QDA

### Fit

In [ ]:
lda = LDA()
lda.fit(X_train, train_df['class'])

In [ ]:
qda = QDA()
qda.fit(X_train, train_df['class'])

### Performance

In [ ]:
pred_train_lda = lda.predict(X_train)
pred_test_lda = lda.predict(X_test)

pred_train_qda = qda.predict(X_train)
pred_test_qda = qda.predict(X_test)

print("Train error rate (LDA):", np.mean(pred_train_lda != train_df['class']))
print("Test error rate (LDA):", np.mean(pred_test_lda != test_df['class']))
print("Train error rate (QDA):", np.mean(pred_train_qda != train_df['class']))
print("Test error rate (QDA):", np.mean(pred_test_qda != test_df['class']))

### Visualization

In [ ]:
x1_min, x1_max = df["x1"].min() - 1, df["x1"].max() + 1
x2_min, x2_max = df["x2"].min() - 1, df["x2"].max() + 1
xx1, xx2 = np.meshgrid(
    np.linspace(x1_min, x1_max, 300),
    np.linspace(x2_min, x2_max, 300),
)
grid = pd.DataFrame({"x1": xx1.ravel(), "x2": xx2.ravel()})
X_grid = design.transform(grid)

pred_lda = lda.predict(X_grid).astype(int).reshape(xx1.shape)
pred_qda = qda.predict(X_grid).astype(int).reshape(xx1.shape)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharex=True, sharey=True)

axes[0].contourf(xx1, xx2, pred_lda, alpha=0.25, cmap="viridis")
axes[0].scatter(df["x1"], df["x2"], c=df["class"], s=5, alpha=0.6, cmap="viridis")
axes[0].set_title("LDA Decision Boundary")
axes[0].set_xlabel("x1")
axes[0].set_ylabel("x2")

axes[1].contourf(xx1, xx2, pred_qda, alpha=0.25, cmap="viridis")
axes[1].scatter(df["x1"], df["x2"], c=df["class"], s=5, alpha=0.6, cmap="viridis")
axes[1].set_title("QDA Decision Boundary")
axes[1].set_xlabel("x1")
axes[1].set_ylabel("x2")

plt.tight_layout()
plt.show()
